# Agentic Demand Forecasting & Inventory Optimization - Colab runner

Runs the whole capstone pipeline (4 agents, baselines, inventory simulation, experiments, figures) in Google Colab.

**How to use:** `Runtime -> Run all`. When asked, upload `agentic-demand-forecasting.zip`.
The default settings use the built-in synthetic data, so everything works without any download.
Switch on the flags in section 1 to run the experiments or the real datasets (M5 / Walmart need a free Kaggle token).

## 1. Choose what to run

In [ ]:
RUN_DEMO = True                 # pipeline on built-in synthetic daily data (~25 s)
RUN_WALMART_SYNTHETIC = True    # weekly mode on Walmart-format synthetic data (~45 s)
RUN_EXPERIMENTS = False         # 5 independent worlds + sensitivity sweeps (~2 min)
RUN_M5 = False                  # real M5 data (needs Kaggle token, see section 6)
RUN_WALMART_REAL = False        # real Walmart weekly data (needs Kaggle token)
USE_NOAA_FOR_M5 = False         # download NOAA temperatures and merge them into M5
M5_FAIR_COMPARISON = True       # after the M5 run: same-fill-rate comparison, sweeps, figures and a draft report

## 2. Get the code

In [ ]:
import os, sys, zipfile, shutil
IN_COLAB = "google.colab" in sys.modules

if not os.path.exists("src/demand_agents"):
    if os.path.exists("agentic-demand-forecasting/src"):
        os.chdir("agentic-demand-forecasting")
    elif IN_COLAB:
        from google.colab import files
        print("Select agentic-demand-forecasting.zip ...")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall(".")
        os.chdir("agentic-demand-forecasting")
    else:
        raise SystemExit("Run this notebook from the project folder (the one containing src/).")
sys.path.insert(0, os.path.abspath("src"))
print("Working in", os.getcwd())

In [ ]:
# Colab already has pandas, numpy, scikit-learn, scipy, matplotlib, plotly. We only need statsmodels (ETS/ARIMA baselines).
import importlib, subprocess
for pkg in ["statsmodels", "yaml"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels" if pkg == "statsmodels" else "pyyaml"], check=True)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from IPython.display import display, Image
print("pandas", pd.__version__)

## 3. Run the agentic pipeline on synthetic daily data

In [ ]:
from demand_agents.config import load_config
from demand_agents.orchestrator import run_pipeline

if RUN_DEMO:
    cfg = load_config()
    state = run_pipeline(cfg, "outputs/latest")

In [ ]:
if RUN_DEMO:
    summ = state.metrics["summary"]
    print("Forecaster chose:", summ["forecaster"]["choice"])
    fm = pd.DataFrame(summ["forecast_metrics"]).pivot(index="model", columns="segment", values="wape").round(3)
    print("\nForecast error (WAPE, lower is better)"); display(fm)
    print("Prediction-interval coverage (90% target)"); display(pd.DataFrame(summ["interval_metrics"]).round(3))
    print("Inventory policies on the same test window"); display(state.inventory_summary.round(3))
    print("Critic quality:", {k: round(v, 3) for k, v in summ["critic"].items()})

### Reasoning traces (what each agent thought, did and decided)

In [ ]:
if RUN_DEMO:
    tr = state.tracer.to_dataframe()
    pd.set_option("display.max_colwidth", 170)
    display(tr[["step", "agent", "kind", "content"]].head(40))

### Interactive forecast explorer

In [ ]:
if RUN_DEMO:
    import ipywidgets as w, plotly.graph_objects as go
    fc = pd.read_csv("outputs/latest/forecasts.csv", parse_dates=["date"])
    series = sorted(fc["series_id"].unique())
    dd = w.Dropdown(options=series, description="Series:"); out = w.Output()

    def draw(_=None):
        s = fc[(fc["series_id"] == dd.value) & fc["split"].isin(["val_check", "test"])]
        fig = go.Figure()
        fig.add_scatter(x=s["date"], y=s["upper"], line=dict(width=0), showlegend=False)
        fig.add_scatter(x=s["date"], y=s["lower"], fill="tonexty", line=dict(width=0), fillcolor="rgba(70,130,180,.2)", name="90% interval")
        fig.add_scatter(x=s["date"], y=s["sales"], name="actual", mode="lines+markers", line=dict(color="black", width=1))
        fig.add_scatter(x=s["date"], y=s["y_hat"], name="forecast")
        if "ets" in s: fig.add_scatter(x=s["date"], y=s["ets"], name="ETS", line=dict(dash="dot"))
        fl = s[s["risk_multiplier"] > 1.2]
        fig.add_scatter(x=fl["date"], y=fl["sales"], mode="markers", name="Critic flag", marker=dict(color="crimson", symbol="diamond-open", size=9))
        fig.update_layout(height=420, margin=dict(t=30), legend=dict(orientation="h"))
        with out:
            out.clear_output(wait=True); fig.show()
    dd.observe(draw, names="value"); display(dd, out); draw()

### Report figures

In [ ]:
if RUN_DEMO:
    subprocess.run([sys.executable, "scripts/make_report_figures.py", "--experiments", "outputs/experiments"], check=True)
    for f in sorted(os.listdir("outputs/figures")):
        print(f); display(Image("outputs/figures/" + f))

## 4. Weekly mode (Walmart-format data)

In [ ]:
if RUN_WALMART_SYNTHETIC:
    cfg_w = load_config("config/walmart.yaml", {"data.source": "walmart_synthetic"})
    state_w = run_pipeline(cfg_w, "outputs/walmart_demo")
    display(state_w.inventory_summary.round(3))

## 5. Experiments: robustness over seeds and sensitivity sweeps

In [ ]:
if RUN_EXPERIMENTS:
    from demand_agents.experiments import run_seed_suite, run_sensitivity
    agg = run_seed_suite(load_config(), range(101, 106), "outputs/experiments/seeds")
    display(agg[agg["kind"] == "inventory"].round(3))
    base = run_pipeline(load_config(), "outputs/experiments/base_run", verbose=False)
    res = run_sensitivity(base, "outputs/experiments")
    for name, df in res.items():
        print(name); display(df.head(12).round(3))

## 6. Real data (optional)

1. On Kaggle open *Settings -> API*. Either click **Generate New Token** (you get a long text token) or **Create Legacy API Key** (you get a `kaggle.json` file). The notebook accepts both and never stores them in the notebook.
2. **M5 only:** open https://www.kaggle.com/competitions/m5-forecasting-accuracy and accept the rules (join the competition), otherwise the download is refused.
3. Set `RUN_M5` / `RUN_WALMART_REAL` to `True` in section 1 and run the cells below.

In [ ]:
if RUN_M5 or RUN_WALMART_REAL:
    import getpass
    # Kaggle now has two kinds of credentials. Either works here:
    #  (a) NEW token: kaggle.com/settings -> API -> "Generate New Token" (a long text string)
    #  (b) OLD key file: same page -> "Create Legacy API Key" (downloads kaggle.json)
    token = getpass.getpass("Paste your Kaggle API token (hidden). Press Enter to upload kaggle.json instead: ").strip()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    else:
        from google.colab import files
        print("Upload kaggle.json"); files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("Kaggle credentials ready")

In [ ]:
if RUN_M5:
    os.makedirs("data/raw/m5", exist_ok=True)
    if not os.path.exists("data/raw/m5/sales_train_evaluation.csv"):
        subprocess.run("kaggle competitions download -c m5-forecasting-accuracy -p data/raw/m5", shell=True, check=True)
        subprocess.run("unzip -o -q data/raw/m5/m5-forecasting-accuracy.zip -d data/raw/m5", shell=True, check=True)
    else:
        print("M5 files already downloaded")
    overrides = {"data.source": "m5", "output_dir": "outputs/m5_pilot"}
    if USE_NOAA_FOR_M5:
        subprocess.run([sys.executable, "scripts/fetch_noaa_weather.py", "--out", "data/raw/weather_m5.csv"], check=True)
        overrides["data.noaa.path"] = "data/raw/weather_m5.csv"
    state_m5 = run_pipeline(load_config(overrides=overrides), "outputs/m5_pilot")
    display(state_m5.inventory_summary.round(3))
    display(pd.DataFrame(state_m5.metrics["summary"]["forecast_metrics"]).pivot(index="model", columns="segment", values="wape").round(3))
    if M5_FAIR_COMPARISON:
        from demand_agents.experiments import run_sensitivity
        res_m5 = run_sensitivity(state_m5, "outputs/experiments_m5")      # re-plans only; forecasts are not recomputed
        print("Average stock needed to reach the same fill rate (lower is better; NaN = never reached):")
        display(res_m5["frontier"].pivot(index="target_fill_rate", columns="policy", values="avg_stock_needed").round(1))
        for script, extra in (("make_report_figures.py", ["--out", "outputs/figures_m5"]), ("build_report.py", ["--out", "outputs/report_m5_draft.md"])):
            subprocess.run([sys.executable, "scripts/" + script, "--results", "outputs/m5_pilot", "--experiments", "outputs/experiments_m5"] + extra, check=True)

In [ ]:
if RUN_WALMART_REAL:
    os.makedirs("data/raw/walmart", exist_ok=True)
    subprocess.run("kaggle datasets download -d yasserh/walmart-dataset -p data/raw/walmart --unzip", shell=True, check=True)
    state_wr = run_pipeline(load_config("config/walmart.yaml"), "outputs/walmart_real")
    display(state_wr.inventory_summary.round(3))

## 7. Draft report and trace-review sheet

In [ ]:
if RUN_DEMO:
    subprocess.run([sys.executable, "scripts/build_report.py", "--experiments", "outputs/experiments"], check=True)
    subprocess.run([sys.executable, "scripts/export_trace_review.py", "--n", "6"], check=True)
    print(open("outputs/report_draft.md").read()[:2500])

## 8. Download everything

In [ ]:
shutil.make_archive("capstone_results", "zip", "outputs")
if IN_COLAB:
    from google.colab import files
    files.download("capstone_results.zip")
else:
    print("Saved capstone_results.zip")

## Optional: the Streamlit dashboard inside Colab
The notebook already shows tables, the interactive explorer and the figures. The full multi-tab dashboard (calibration, what-if planner, traces)
is easiest to run on a laptop: `pip install -e ".[dashboard]"` then `streamlit run dashboard/app.py`.